# Swarm obstacle avoidance: RL training on Kaggle

Runs the same training as `scripts/rl_train.py` on the laptop, but longer and with several seeds.
Before running:
1. Add the dataset **swarm-rl-code** (the zip from `kaggle/swarm-rl-code.zip`) to this notebook (Add Input).
2. Settings: **Internet on** (needs a phone-verified account) so `pip` can install Stable-Baselines3.
3. Accelerator: **None (CPU)** is enough; the simulator runs on the CPU and the network is small.

Outputs land in `/kaggle/working/` (download `results.zip` at the end).

In [ ]:
!pip install -q "stable-baselines3==2.9.0" "gymnasium==1.3.0"
import os, glob, shutil, zipfile, subprocess
src = glob.glob('/kaggle/input/**/swarm-rl-code.zip', recursive=True) or glob.glob('/kaggle/input/**/src', recursive=True)
print(src)

In [ ]:
os.makedirs('/kaggle/working/code', exist_ok=True)
if src and src[0].endswith('.zip'):
    zipfile.ZipFile(src[0]).extractall('/kaggle/working/code')
else:  # Kaggle unpacked the zip already
    shutil.copytree(os.path.dirname(src[0]), '/kaggle/working/code', dirs_exist_ok=True)
os.chdir('/kaggle/working/code')
print(os.listdir('.'), os.cpu_count(), 'CPUs')

In [ ]:
# Two seeds in parallel (one per 2 CPUs). 20 M steps each takes a few hours; lower STEPS for a quick try.
STEPS, SEEDS = 20_000_000, [1, 2]
env = dict(os.environ, PYTHONPATH='src', OMP_NUM_THREADS='2')
procs = [subprocess.Popen(['python', 'scripts/rl_train.py', '--steps', str(STEPS), '--seed', str(s), '--n-sims', '3',
                           '--run', f'/kaggle/working/run_seed{s}'], env=env,
                          stdout=open(f'/kaggle/working/train_seed{s}.log', 'w'), stderr=subprocess.STDOUT) for s in SEEDS]
for p in procs:
    p.wait()
print([p.returncode for p in procs])

In [ ]:
!tail -n 3 /kaggle/working/train_seed*.log; cat /kaggle/working/run_seed*/eval.jsonl | tail -n 6

In [ ]:
# Held-out comparison (same protocol as on the laptop) for the best policy of every seed
for s in SEEDS:
    subprocess.run(['python', 'scripts/rl_eval.py', '--policy', f'/kaggle/working/run_seed{s}/best_policy.npz',
                    '--episodes', '30', '--jobs', '4', '--out', f'/kaggle/working/eval_seed{s}'],
                   env=dict(os.environ, PYTHONPATH='src'), check=True)
    print(open(f'/kaggle/working/eval_seed{s}/summary.md').read())

In [ ]:
shutil.make_archive('/kaggle/working/results', 'zip', '/kaggle/working', '.')  # includes runs and evals
print('download /kaggle/working/results.zip')